# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)
features = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]
print(df.shape, "declining rate:", round(df["label_declining"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9) declining rate: 0.329


In [4]:
b = df.dropna(subset=["early_avg_position", "early_ctr"]).copy()
b = b[b["early_avg_position"] <= 20].copy()
b["pos_bucket"] = pd.cut(b["early_avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)
sums = b.groupby("pos_bucket")[["early_clicks", "early_impressions"]].sum()
b["peer_ctr"] = b["pos_bucket"].map(sums["early_clicks"] / sums["early_impressions"])
b["baseline_score"] = (b["peer_ctr"] * b["early_impressions"] - b["early_clicks"]).clip(lower=0)

df["baseline_score"] = 0.0
df.loc[b.index, "baseline_score"] = b["baseline_score"]
print("pages with a baseline score:", int((df["baseline_score"] > 0).sum()))

pages with a baseline score: 1181


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I compare three models and my ML-07 rule baseline. Logistic regression is the simplest: five inputs, readable coefficients, hard to overfit. A decision tree with depth 3 gives rules a person can read, like my baseline. A random forest can catch interactions between signals. I did not use gradient boosting: with about 2,200 pages and five features, extra complexity is unlikely to help, and a model I can't explain is not worth more than a rule I can. All inputs are early-window columns (1–15 March); the label (16–31 March) is never an input.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I use grouped 5-fold cross-validation with client_hash_id as the group, so no client appears in both training and test data. Every page is scored exactly once by a model that never saw its client, and the baseline is scored on the same pages with the same metric (Precision@50, plus AUC and Precision@100). Features come from 1–15 March and the label from 16–31 March, so nothing from the label window reaches an input. Limitation: this is a split by client within one month, not a split across time.

In [7]:
from sklearn.model_selection import GroupKFold

X = df[features].fillna(0)
y = df["label_declining"]
groups = df["client_hash_id"]
gkf = GroupKFold(n_splits=5)

for k, (tr, te) in enumerate(gkf.split(X, y, groups)):
    overlap = set(groups.iloc[tr]) & set(groups.iloc[te])
    print(f"fold {k}: train {len(tr)}, test {len(te)}, clients in both: {len(overlap)}")

fold 0: train 1750, test 441, clients in both: 0
fold 1: train 1753, test 438, clients in both: 0
fold 2: train 1753, test 438, clients in both: 0
fold 3: train 1754, test 437, clients in both: 0
fold 4: train 1754, test 437, clients in both: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=0, n_jobs=-1),
}

oof = {}
for name, m in models.items():
    p = np.zeros(len(df))
    for tr, te in gkf.split(X, y, groups):
        m.fit(X.iloc[tr], y.iloc[tr])
        p[te] = m.predict_proba(X.iloc[te])[:, 1]
    oof[name] = p
oof["Baseline rule (ML-07)"] = df["baseline_score"].values
print("done:", list(oof))

done: ['Logistic regression', 'Decision tree (depth 3)', 'Random forest', 'Baseline rule (ML-07)']


In [10]:
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, score, k):
    idx = np.argsort(-score, kind="stable")[:k]
    return y_true.values[idx].mean()

rows = []
for name, p in oof.items():
    rows.append({"model": name,
                 "AUC": roc_auc_score(y, p),
                 "Precision@50": precision_at_k(y, p, 50),
                 "Precision@100": precision_at_k(y, p, 100)})
table = pd.DataFrame(rows).round(3)
print("base rate (random picking):", round(y.mean(), 3))
table

base rate (random picking): 0.329


,model,AUC,Precision@50,Precision@100
0,Logistic regression,0.560,0.38,0.39
1,Decision tree (depth 3),0.545,0.12,0.27
2,Random forest,0.558,0.24,0.27
3,Baseline rule (ML-07),0.570,0.38,0.42


On the same pages and the same grouped split, the best model is ____ with Precision@50 of ____ and AUC of ____, against ____ and ____ for my baseline rule (base rate 0.329). With only 50 pages, Precision@50 moves about 7 points by chance, so a gap smaller than that is not clear evidence. (Say whether the models clearly beat the baseline or not, and whether the extra complexity was worth it.)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
from sklearn.inspection import permutation_importance

tr, te = next(GroupKFold(n_splits=5).split(X, y, groups))
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=0, n_jobs=-1).fit(X.iloc[tr], y.iloc[tr])
pi = permutation_importance(rf, X.iloc[te], y.iloc[te], scoring="roc_auc", n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=features).sort_values(ascending=False).round(4)

,0
early_ctr,0.1065
early_avg_position,0.0568
early_active_days,0.0182
early_clicks,0.0072
early_impressions,0.0034


In [13]:
best = "Random forest"
s = pd.Series(oof[best], index=df.index)
top = df.loc[s.sort_values(ascending=False).head(50).index]
print("top 50: declined =", int(top["label_declining"].sum()), ", did not decline =", int((1 - top["label_declining"]).sum()))
top.groupby("label_declining")[features].median().round(3)

top 50: declined = 12 , did not decline = 38


,early_impressions,early_clicks,early_ctr,early_avg_position,early_active_days
label_declining,,,,,
0,3242.0,3.0,0.001,7.374,15.0
1,4092.0,3.0,0.001,5.741,15.0


In [14]:
from sklearn.tree import export_text
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0).fit(X, y)
print(export_text(tree, feature_names=features))

|--- early_ctr <= 0.00
|   |--- early_active_days <= 13.50
|   |   |--- early_impressions <= 107.50
|   |   |   |--- class: 0
|   |   |--- early_impressions >  107.50
|   |   |   |--- class: 0
|   |--- early_active_days >  13.50
|   |   |--- early_impressions <= 492.50
|   |   |   |--- class: 0
|   |   |--- early_impressions >  492.50
|   |   |   |--- class: 1
|--- early_ctr >  0.00
|   |--- early_impressions <= 69.00
|   |   |--- class: 0
|   |--- early_impressions >  69.00
|   |   |--- early_active_days <= 13.50
|   |   |   |--- class: 0
|   |   |--- early_active_days >  13.50
|   |   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.